## Import Class

In [ ]:
from exafs_workflow import *
#%matplotlib widget

## Larch Parameters

In [ ]:
LARCH_PARAMS = {

    "pre_edge": dict(
        pre1=-200,
        pre2=-30,
        nvict=1,
        npre=1,
        nnorm=1,
        norm1=150,
        norm2=-1
    ),

    "autobk": dict(
        rbkg=1.00,     # CRITICAL CHANGE
        kmin=0,     # suppress low-k junk
        kmax=10,
        kweight=2,
        clamp_lo=2.0,
        clamp_hi=20.0
    ),

    # k → R Fourier transform
    "xftf": dict(
        kmin=2,
        kmax=10,
        kweight=2,
        dk=0.25,
        window="Kaiser-Bessel",
        rmax_out=6
    ),

    # R → k back transform (shell filtering etc.)
    "xftr": dict(
        rmin=1.0,
        rmax=5,
        kweight=2,
        dr=0.05,
        window='Hanning'
    )
}


## Single file data reduction and analysis

In [ ]:
base_dir = project_number("36G12627")
#If not in a project use directory link
# base_dir = "/beamlinedata/BIOXAS-SPECTROSCOPY/old_data/bioxas-m/AcquamanMainData/users/Ponomarenko/exportData/31-10797/"

In [ ]:
## Single file data reduction and analysis
exafs = EXAFSAll()
filename = 'OP_MTR3_grain_16k_yp985_zp482_ZnK_23.dat'
# filename = 'EL_rbc_AsSe_AsK_1_EXAFS_1.dat'
full_path = os.path.join(base_dir, filename)
aligned_data, e0 = exafs.align_multiple(larch_params=LARCH_PARAMS,
    files= {"Scan1": full_path},                       # dict like {"Scan1": "/path/to/file1", ...}
    inb_filter="AsKa1_spectra",  # Excitation edge starting with the element name 
    reference=None,
    summed_groups=None, # None for single files # summed_groups for sum files,    # All for all files
    ion_chamber="I0",
    channel_side="OutB",  # inB or OutB channel for analysis
    auto_rbkg=False,   # default; can omit
    min_snr_post = 21.0,      # edge_jump / rms_post_residual cutoff level for chosing good channels
    plot_mode="aligned_mu",            # returns (aligned_data, e0)
    prompt_save=False,   # no export here
    deglitch_mode='none', #'pfy',  # 'pfy'  | 'both' | 'none'.
    deglitch_region="exafs",  # xanes, exafs or custom
    # deglitch_xrange=(8330.0, 8390.0),  # tuple OR dict of named ranges
)


In [ ]:
## Single file data reduction and analysis
exafs = EXAFSAll()
base_dir = "/beamlinedata/BIOXAS-SPECTROSCOPY/old_data/bioxas-m/AcquamanMainData/users/Ponomarenko/exportData/31-10797/"
filename = 'EL_rbc_AsSe_AsK_1_EXAFS_1.dat'
full_path = os.path.join(base_dir, filename)
aligned_data, e0 = exafs.align_multiple(larch_params=LARCH_PARAMS,
    files= {"Scan1": full_path},                       # dict like {"Scan1": "/path/to/file1", ...}
    inb_filter="AsKa1_spectra",  # Excitation edge starting with the element name 
    reference=None,
    summed_groups=None, # None for single files # summed_groups for sum files,    # All for all files
    ion_chamber="I0",
    channel_side="OutB",  # inB or OutB channel for analysis
    auto_rbkg=False,   # default; can omit
    min_snr_post = 21.0,      # edge_jump / rms_post_residual cutoff level for chosing good channels
    plot_mode="aligned_mu",            # returns (aligned_data, e0)
    prompt_save=False,   # no export here
    deglitch_mode='none', #'pfy',  # 'pfy'  | 'both' | 'none'.
    deglitch_region="exafs",  # xanes, exafs or custom
    # deglitch_xrange=(8330.0, 8390.0),  # tuple OR dict of named ranges
)


In [ ]:
# Optional print using safe labels
# If your dataset exposes 32 InB/OutB channels matching inb_filter,
# this makes a 6×6 grid (36 slots), with the 33rd for sum(good)
exafs.plot_good_channels('Scan1', ncols=6, normalize_each=True)


In [ ]:
# per-label metrics DataFrame
exafs.metrics["Scan1"].head(34)

In [ ]:
# Optional print using safe labels
exafs.print_e0_values()

In [ ]:
# export everything
csv_path = exafs.save_metrics_csv("/paths/metrics_all.csv")
print("Saved:", csv_path)

In [ ]:
#plot mu for single data and the other reduction data
exafs.plot_mu(
        xlim=(11600,12800),         
        show_raw = True,
        show_pre_edge = True,
        show_post_edge = True,
        show_bkg = True,
        show_bkgsub = False,
        show_norm = False,
        show_flat = False,
        offset_step = 0.0)


In [ ]:
#plot mu for single data and the other reduction data
exafs.plot_mu(
        xlim=(11600,12800),         
        show_raw = False,
        show_pre_edge = False,
        show_post_edge = False,
        show_bkg = False,
        show_bkgsub = False,
        show_norm = True,
        show_flat = True,
)


In [ ]:
# plotting k space for single data to save the data include directory
exafs.plot_chi_k('Scan1')#, outdir="Path")

In [ ]:
# plotting R space for single data to save the data include directory
exafs.plot_chi_r('Scan1')#, outdir="/Paths/ND")

In [ ]:
# plotting Wavelet space for single data to save the data include directory
exafs.plot_wavelet('Scan1')#, outdir="Path")

# Multiple files data reduction and analysis

In [ ]:
exafs = EXAFSAll()
# base_dir = project_number("########")
base_dir = "/beamlinedata/BIOXAS-SPECTROSCOPY/old_data/bioxas-m/AcquamanMainData/users/Ponomarenko/exportData/31-10797/"
# --- Run the preparation ---
files, filenames_sorted, groups_by_token, label_map, summed_groups = prepare_inputs_any(
    base_dir=base_dir,
    # exclude_glob = ['*'],   # Exclude data files you want to plot
    include_glob = ['EL_rbc_AsSe_AsK_1_EXAFS_*'],  # Include data files you want to plot
    scan_label_prefix=None,             # kept in signature; labels ignore prefixes by design now
    prefer_energy_first=True,           # energy groups listed first (if any)
    reset_scan_numbers_per_group=False,  # GLOBAL indices (recommended for EXAFS workflows)
    preview_input=True,
)

In [ ]:
# Align and Sum (Multi-file)
aligned_data, e0 = exafs.align_multiple(larch_params=LARCH_PARAMS,
    files=files,                       # dict like {"Scan1": "/path/to/file1", ...}
    # reference=8366.0002,
    inb_filter="AsKa1",   # Excitation edge starting with the element name 
    ion_chamber="I0",     # ion chamber for noralization
    summed_groups=summed_groups,    # None for single files
    plot_mode="As_k-sums_only",            # returns (aligned_data, e0)
    channel_side="OutB",
    auto_rbkg=False,   # default; can omit
    min_snr_post = 21.0,      # edge_jump / rms_post_residual cutoff level for chosing good channels
    prompt_save=False,   # no export here
    deglitch_mode='none', #'pfy',  # 'pfy'  | 'both' | 'none'
    deglitch_region="custom",  # 'xanes'| 'exafs' | 'custom'
    deglitch_xrange=(12000.0, 12700.0),  # tuple OR dict of named ranges
)


# Lit the sample group name
samples = sample_labels(label_map)

# Optional print using safe labels
exafs.print_e0_values()

In [ ]:
# Plot (Multi-file)
exafs.plot_mu(
        # xlim=(6400,7100),         
        show_raw = False,
        show_pre_edge = False,
        show_post_edge = False,
        show_bkg = False,
        show_bkgsub = False,
        show_norm = False,
        show_flat = True,
        offset_step = 1.0)

In [ ]:
# Plotting k-space for group data
exafs.plot_chi_k(labels=samples, show_raw = False,stack="subplots", sharey=False) #, outdir="/Paths/ND")

In [ ]:
# Plotting k-space for group data
exafs.plot_chi_r(labels=samples, stack="subplots", sharey=False) #, outdir="/Paths/ND")



In [ ]:
# Plotting wavelet-space for group data
exafs.plot_wavelet(label=samples, stack=True)#, outdir="/Paths/ND")

In [ ]:
# Save Sum or single normalized data using your custom labels in filenames
exafs.save_signals(folder="Paths", y="norm", summed_only=True, use_display_label=True)

In [ ]:
# Save Sum or single  flat data for sums using your custom labels in filenames
exafs.save_signals(folder="Paths", y="flat", summed_only=True, use_display_label=True)
